# Fine tuning DBSCAN pseudo-label weight caps

Hold confidence thresholds at positive 0.60 and negative 0.10, and hold the current DBSCAN density, 150 embeddings, scalers, and logistic regression C=1000 fixed. Independently cross **class-weight budgets 35%, 40%, 45%, 50%, 55%, 60%, 65%, 70%** with **per-generated-label caps 0.15 to 0.35 in steps of 0.025**. This gives 72 combinations plus a labeled-only control.

The class budget limits the added weight for each generated class relative to the known weight of that same class. The per-example cap limits each pseudo-label's raw weight. Known labels retain raw weight 1; generated positive and negative total weights preserve the known-label class ratio. The smaller constraint wins, so different nominal budgets may yield identical weights. Record the active constraint and distinct actual datasets as well as scores. No Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
POSITIVE_THRESHOLDS = [0.6]
NEGATIVE_THRESHOLDS = [0.1]
CLASS_BUDGETS = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
PER_LABEL_CAPS = [0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350]
from sklearn.cluster import DBSCAN
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"dbscan_weight_fine_tuning"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
reference_dir=DATA/"experiments"/"additional_methods"
reference_record=json.loads((reference_dir/"submission_manifest.json").read_text())
reference_model=joblib.load(reference_dir/reference_record["model_file"])
reference_probability=predict_reason_models(reference_model,X_valid)
reference_metrics=save_predictions("dbscan_current_reference",reference_probability)
np.testing.assert_allclose(reference_metrics["sample_f1"],reference_record["validation_sample_f1"],atol=1e-14)
density_packet=joblib.load(reference_dir/"labeling_models/dbscan_m5_q25.joblib")
density=density_packet["model"]
assert density.min_samples==5 and density.eps>0
np.testing.assert_array_equal(density_packet["scaler"].mean_,cluster_scaler.mean_)
# Check that the persisted clustering corresponds to this exact training input.
with threadpool_limits(limits=1):
    density_replay=DBSCAN(eps=density.eps,min_samples=density.min_samples,metric="euclidean",algorithm="brute",n_jobs=1).fit(XC)
np.testing.assert_array_equal(density_replay.labels_,density.labels_)
ids=density.labels_;clusters=np.unique(ids[ids>=0])
membership=np.column_stack([ids==c for c in clusters]).astype(float)
cluster_support=membership[known].sum(axis=0)
positive_counts=membership[known].T@original_y[known]
prior=original_y[known].mean(axis=0)
rates=(positive_counts+LABEL_RULE["prior_strength"]*prior)/(cluster_support[:,None]+LABEL_RULE["prior_strength"])
probability=membership@rates
support=membership@cluster_support
cluster_details=pd.DataFrame({"cluster":clusters,"training_trials":membership.sum(axis=0),"known_support":cluster_support})
for j,r in enumerate(REASONS):cluster_details[f"probability_{r}"]=rates[:,j]
cluster_details.to_csv(OUT/"clusters/fixed_dbscan.csv",index=False)
joblib.dump(density_packet,OUT/"labeling_models/fixed_dbscan.joblib",compress=3)
reference_frame=pd.read_csv(reference_dir/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
for q in [reference_dir/reference_record["model_file"],reference_dir/"datasets/dbscan_m5_q25.csv",reference_dir/"labeling_models/dbscan_m5_q25.joblib",reference_dir/reference_record["submission_file"]]:
    protected_hashes[str(q.relative_to(DATA))]=hashlib.sha256(q.read_bytes()).hexdigest()
print(f"Fixed DBSCAN: eps={density.eps:.17g}, min_samples=5; reference sample F1={reference_metrics['sample_f1']:.6f}")


Fixed DBSCAN: eps=0.00012142366141394104, min_samples=5; reference sample F1=0.910256


In [4]:
ROWS=[];MODELS={};RULES={};CONSTRAINT_ROWS=[]

def generate_weighted_targets(cap,budget):
    rule={**LABEL_RULE,"maximum_generated_weight":cap,"maximum_class_fraction":budget}
    targets=np.full_like(original_y,-100);weights=np.zeros(original_y.shape)
    targets[known]=original_y[known];weights[known]=1.0
    eligible=(~known)&(ids>=0)&(support>=rule["minimum_support"])
    constraints=[]
    for j,r in enumerate(REASONS):
        pos=eligible&(probability[:,j]>=rule["positive_threshold"])
        neg=eligible&(probability[:,j]<=rule["negative_threshold"])
        np_,nn=int(pos.sum()),int(neg.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not np_ or not nn or not cap or not budget:continue
        bounds={"class_budget":budget,"positive_example_cap":cap*np_/kp,"negative_example_cap":cap*nn/kn}
        active=min(bounds,key=bounds.get);fraction=bounds[active]
        targets[pos,j]=1;targets[neg,j]=0
        weights[pos,j]=fraction*kp/np_;weights[neg,j]=fraction*kn/nn
        assert weights[~known,j].max()<=cap+1e-12
        np.testing.assert_allclose(weights[pos,j].sum()/kp,weights[neg,j].sum()/kn,atol=1e-12)
        assert weights[pos,j].sum()/kp<=budget+1e-12
        constraints.append({"reason":r,"positive_candidates":np_,"negative_candidates":nn,
                            **bounds,"actual_class_fraction":fraction,"active_constraint":active,
                            "positive_weight_each":fraction*kp/np_,"negative_weight_each":fraction*kn/nn})
    assert (weights[(ids<0)&~known]==0).all()
    return targets,weights,rule,constraints

def run_configuration(cap,budget):
    name=f"cap{cap:g}_budget{budget:g}".replace(".","p") if cap else "known_only_control"
    y,w,rule,constraints=generate_weighted_targets(cap,budget)
    dataset=save_dataset(name,y,w,np.where((ids>=0)[:,None],probability,np.nan),ids,support)
    RULES[name]=rule
    CONSTRAINT_ROWS.extend({"experiment":name,"maximum_generated_weight":cap,"maximum_class_fraction":budget,**row} for row in constraints)
    is_reference=cap==0.25 and budget==0.5
    if is_reference:
        np.testing.assert_array_equal(y,reference_frame[REASONS].to_numpy(dtype=int))
        np.testing.assert_allclose(w,reference_frame[[f"weight_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-14)
    with threadpool_limits(limits=1):model=fit_reason_models(name,"logistic",1000)
    model["label_rule"]=rule.copy();model["density_model_file"]="labeling_models/fixed_dbscan.joblib"
    probabilities=predict_reason_models(model,X_valid)
    if is_reference:np.testing.assert_allclose(probabilities,reference_probability,rtol=1e-8,atol=1e-10)
    if not cap:np.testing.assert_allclose(probabilities,baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
    metrics=save_predictions(name,probabilities)
    joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3);MODELS[name]=model
    distance=abs(cap-0.25)/0.025+abs(budget-0.5)/0.05 if cap else 100
    ROWS.append({"experiment":name,"maximum_generated_weight":cap,"maximum_class_fraction":budget,
                 "generated_trials":dataset["generated_trials"],"distance_from_reference_settings":distance,
                 "target_weight_sha256":hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest(),
                 **metrics,"delta_vs_reference":metrics["sample_f1"]-reference_metrics["sample_f1"],"kaggle_score":np.nan})

run_configuration(0.0,0.0)
for budget in CLASS_BUDGETS:
    for cap in PER_LABEL_CAPS:run_configuration(cap,budget)
    subset=pd.DataFrame(ROWS).query("maximum_class_fraction==@budget")
    print(f"Finished class budget {budget:.0%}: {len(subset)} configurations; best sample F1={subset.sample_f1.max():.6f}",flush=True)
assert len(ROWS)==73


Finished class budget 35%: 9 configurations; best sample F1=0.910256


Finished class budget 40%: 9 configurations; best sample F1=0.910256


Finished class budget 45%: 9 configurations; best sample F1=0.910256


Finished class budget 50%: 9 configurations; best sample F1=0.910256


Finished class budget 55%: 9 configurations; best sample F1=0.910256


Finished class budget 60%: 9 configurations; best sample F1=0.910256


Finished class budget 65%: 9 configurations; best sample F1=0.910256


Finished class budget 70%: 9 configurations; best sample F1=0.910256


In [5]:
comparison=pd.DataFrame(ROWS)
ranked=comparison.sort_values(["sample_f1","distance_from_reference_settings","experiment"],ascending=[False,True,True],kind="stable")
best=ranked.iloc[0]
comparison.to_csv(OUT/"comparison.csv",index=False)
pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
constraints=pd.DataFrame(CONSTRAINT_ROWS)
constraints.to_csv(OUT/"active_constraints.csv",index=False)
cap_only=ranked.query("maximum_class_fraction==0.5").sort_values("maximum_generated_weight")
budget_only=ranked.query("maximum_generated_weight==0.25").sort_values("maximum_class_fraction")
cap_only.to_csv(OUT/"per_label_cap_comparison.csv",index=False)
budget_only.to_csv(OUT/"class_budget_comparison.csv",index=False)
print("Best:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
print("Unique target/weight tables:",comparison.target_weight_sha256.nunique(),"of",len(comparison))
print("Active constraints:",constraints.active_constraint.value_counts().to_dict())
display(cap_only[["maximum_generated_weight","maximum_class_fraction","generated_trials","sample_f1","macro_f1","delta_vs_reference"]])


Best: cap0p25_budget0p5 sample F1: 0.9102564102564102 macro F1: 0.811421460284173
Unique target/weight tables: 10 of 73
Active constraints: {'positive_example_cap': 72}


,maximum_generated_weight,maximum_class_fraction,generated_trials,sample_f1,macro_f1,delta_vs_reference
28,0.150,0.5,133,0.897436,0.801106,-0.012821
29,0.175,0.5,133,0.897436,0.803884,-0.012821
30,0.200,0.5,133,0.897436,0.803884,-0.012821
31,0.225,0.5,133,0.897436,0.803884,-0.012821
32,0.250,0.5,133,0.910256,0.811421,0.000000
33,0.275,0.5,133,0.897436,0.803884,-0.012821
34,0.300,0.5,133,0.897436,0.803884,-0.012821
35,0.325,0.5,133,0.897436,0.803884,-0.012821
36,0.350,0.5,133,0.897436,0.803884,-0.012821


In [6]:
from sklearn.base import clone
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
selected_probability=predict_reason_models(selected,X_valid)
np.testing.assert_allclose(selected_probability,predict_reason_models(MODELS[best.experiment],X_valid),rtol=1e-12,atol=1e-12)
frame=pd.read_csv(OUT/"datasets"/f"{best.experiment}.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()
replay={**selected,"estimators":[]}
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy())
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_reason_models(replay,X_valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(selected_probability)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

candidate=None
if best.sample_f1>reference_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    predicted,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_logistic_C1000_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"test_rows":len(output),"test_fallback_trials":int(empty.sum()),
               "submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib",
               "model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"features":FEATURES,"positive_thresholds":POSITIVE_THRESHOLDS,"negative_thresholds":NEGATIVE_THRESHOLDS,
          "per_label_caps":PER_LABEL_CAPS,"class_budgets":CLASS_BUDGETS,"base_label_rule":LABEL_RULE,"best_label_rule":RULES[best.experiment],
          "density_eps":float(density.eps),"density_min_samples":5,"classifier_C":1000,
          "classifier_scaler_fit_rows":157,"clustering_scaler_fit_rows":1171,"fitting_weight_total_per_reason":157,
          "prediction_cutoff":0.5,"nonempty_fallback":True,"validation_rows":78,"validation_used_for_fit":False,
          "configurations":len(comparison),"unique_target_weight_tables":int(comparison.target_weight_sha256.nunique()),
          "reference_sample_f1":reference_metrics["sample_f1"],"reference_macro_f1":reference_metrics["macro_f1"],
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob('*.csv'))},
          "reported_kaggle_best":0.87393,"new_kaggle_scores":None,
          "selection":"sample-wise F1; ties favor settings closest to reference, then experiment name",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__}}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
print("Model reload, independent saved-dataset refit, original-setting reproduction, and protected-input checks passed.")
print("Candidate:",candidate)


Model reload, independent saved-dataset refit, original-setting reproduction, and protected-input checks passed.
Candidate: None


In [7]:
def result_table(rows):
    lines=["| Per-label cap | Class-weight budget | Generated trials | Sample F1 | Macro F1 | Change |",
           "|---:|---:|---:|---:|---:|---:|"]
    for row in rows.itertuples(index=False):
        lines.append(f"| {row.maximum_generated_weight:g} | {row.maximum_class_fraction:.0%} | {row.generated_trials} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_reference:+.6f} |")
    return "\n".join(lines)
selected_counts=pd.DataFrame(LABEL_COUNTS).query("dataset==@best.experiment")
selected_constraints=constraints.query("experiment==@best.experiment")
lines=["# Fine tuning DBSCAN pseudo-label weight caps","",
       "72 weight-cap/budget combinations plus a labeled-only control were evaluated on the same 78 validation trials. "
       "Positive confidence threshold 0.60, negative threshold 0.10, clustering, 150 embeddings, training-only scalers, "
       "classifier C=1000, prediction cutoff 0.5, and nonempty fallback remain fixed.","",
       "Class budgets: 35%, 40%, 45%, 50%, 55%, 60%, 65%, 70%. "
       "Per-label caps: 0.15, 0.175, 0.20, 0.225, 0.25, 0.275, 0.30, 0.325, 0.35. "
       "The two caps vary independently. Known labels keep raw weight 1, generated positive/negative total weights preserve "
       "the known class ratio, and fitting weights are normalized to total 157 per reason.","",
       f"Reference sample F1: **{reference_metrics['sample_f1']:.6f}**, macro F1: **{reference_metrics['macro_f1']:.6f}**.",
       f"Best configuration: **{best.experiment}**; sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**; "
       f"sample F1 change **{best.delta_vs_reference:+.6f}**.",
       f"The 73 configurations produced **{comparison.target_weight_sha256.nunique()} distinct target/weight tables**, including the control.","",
       "## Per-label cap changes at the original 50% class budget","",result_table(cap_only),"",
       "## Class-budget changes at the original 0.25 per-label cap","",result_table(budget_only),"",
       "## Active constraints","",
       f"Constraint counts across accepted reasons/configurations: {constraints.active_constraint.value_counts().to_dict()}.",
       "A nominal cap only changes fitting when it is the smallest bound. For the current 8 generated Efficacy positives "
       "and 17 known positives, a per-example cap of 0.25 limits the added positive weight to 2, or about 11.76% of known "
       "positive weight. Thus a class budget of 35–70% is already above that bound. The negative weights are adjusted to "
       "preserve the same added fraction for the negative class.","",
       "## All combinations","",result_table(ranked),"",
       "## Selected generated labels","",
       "| Reason | Generated positives | Generated negatives | Positive raw weight total | Negative raw weight total |",
       "|---|---:|---:|---:|---:|"]
for row in selected_counts.itertuples(index=False):
    lines.append(f"| {row.reason} | {row.generated_positives} | {row.generated_negatives} | {row.generated_positive_weight:.6f} | {row.generated_negative_weight:.6f} |")
lines.extend(["","## Verification","",
       "The original configuration reproduces the previous targets, weights, and probabilities. The zero-weight control "
       "reproduces the labeled-only baseline. Known targets remain unchanged, DBSCAN noise stays unlabeled, and validation "
       "never enters clustering or fitting. The selected model was reloaded and independently refitted from its saved "
       "dataset with matching validation probabilities; protected earlier artifacts retain their checksums.","",
       "Every dataset, label audit, classifier, validation prediction, configuration, and checksum is saved locally. "
       "Selection uses sample-wise F1; ties favor the original settings. Validation tuning is exploratory and does not "
       "establish independent test improvement. No Kaggle submission was used; no commit or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "The primary validation metric did not improve; retain the existing DBSCAN candidate.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Saved summary:",OUT/"SUMMARY.md")
print(result_table(cap_only))


Saved summary: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/experiments/dbscan_weight_fine_tuning/SUMMARY.md
| Per-label cap | Class-weight budget | Generated trials | Sample F1 | Macro F1 | Change |
|---:|---:|---:|---:|---:|---:|
| 0.15 | 50% | 133 | 0.897436 | 0.801106 | -0.012821 |
| 0.175 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.2 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.225 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.25 | 50% | 133 | 0.910256 | 0.811421 | +0.000000 |
| 0.275 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.3 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.325 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |
| 0.35 | 50% | 133 | 0.897436 | 0.803884 | -0.012821 |


## Sensitivity of the nearest settings

The nearest per-label caps, **0.225** and **0.275**, each change the predicted reason set for only **one of the 78 validation trials** relative to 0.25. In both comparisons that trial uses the nonempty fallback: small changes in the Efficacy probability change which reason has the highest probability when all probabilities are below 0.5. The current cap wins this comparison, but this one-trial difference is limited evidence for choosing a precise weight and does not establish that 0.25 will generalize better.